# Test Squidpy

## Data Import

In [ ]:
import squidpy as sq
import pandas as pd
import scanpy as sc
import numpy as np

In [ ]:
dir_notebook = '/media/volume/volume_spatial/hugo/notebook'
name_dir = "circa-SD"
adata = sc.read_h5ad(f'{dir_notebook}/h5ad/{name_dir}/{name_dir}_final.h5ad.gz')
# adata_bis = sc.read_h5ad(r'..\..\h5ad\run3-SC\run3-SC_clusters.h5ad.gz')


In [ ]:
# adata = adata[adata.obs['sample'] == 'SD1-ZT01']
adata = adata[adata.obs['run'] == "circa4"] 
# adata_bis = adata_bis[adata_bis.obs['sample'] == '3161-3']
# adata = adata[adata.obs['region_automap_name']=='SCH']
# adata.obsm['spatial'] = adata.obsm['coord_xy']

In [ ]:
adata.obsm['spatial'] = adata_bis.obsm['spatial'] 

In [ ]:
adata_sub = adata[adata.obs["region_automap_name"]=='CTX']

In [ ]:
adata_sub.obs.sample()

## Spatial correlation

In [ ]:
sq.gr.spatial_neighbors(adata_sub)

In [ ]:
sq.gr.nhood_enrichment(adata_sub, cluster_key="cell_type_final")

In [ ]:
sq.pl.nhood_enrichment(adata_sub, cluster_key="cell_type_final")

## Moran

In [ ]:
sq.gr.spatial_neighbors(adata_sub, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(
    adata_sub,
    mode="moran",
    n_perms=100,
    n_jobs=5,
)
adata_sub.uns["moranI"].head(15)

In [ ]:
sq.pl.spatial_scatter(
    adata_sub,
    library_id="spatial",
    color=[
        "Gfap",
        "Vip",
        "Arc"
    ],
    shape=None,
    size=2,
    img=False,
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=[
        "Tlr1",
        "Tlr6",
        "Tlr3"
    ],
    shape=None,
    size=2,
    img=False,
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=[
        "Gpr88",
        "Ppp3r1",
        "Foxj1"
    ],
    shape=None,
    size=2,
    img=False,
)

In [ ]:
sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    color=[
        "Sostdc1",
    ],
    shape=None,
    size=2,
    img=False,
)

## LigRec

In [ ]:
adata_sub.raw = adata_sub.copy()

In [ ]:
adata_sub.obs['cell_type_final'].value_counts(normalize=False)

In [ ]:
sc.pp.subsample(adata_sub, fraction = 0.1) ### Will automatically replace adata_sub everytime you run it. Be carefull

In [ ]:
sq.gr.ligrec(adata_sub,
             "cell_type_final",
            #  clusters = ("L6 CT CTX Glut", "Vip Gaba"),
             use_raw=True,
             complex_policy = "all",
            #  threshold = 0.5,
             corr_method = 'fdr_bh',
             alpha =0.01,
             n_jobs = 2,
             n_perms = 10000)

In [ ]:
adata_sub.uns['cell_type_final_ligrec'].keys()

In [ ]:
adata_sub.uns['cell_type_final_ligrec']['pvalues'].to_csv('data/LigRec_CTX_circa4_pvalues.csv')
#[('Oligodendrocyte','Oligodendrocyte')]

In [ ]:
sq.pl.ligrec(adata_sub,
            source_groups="Sst Gaba",
            # target_groups= "Microglia",
            cluster_key='cell_type_final',
            remove_nonsig_interactions=True,
            alpha = 0.001,
            dendrogram= None,
            pvalue_threshold = 0.001,
            means_range = (0.5,1.9)
             )
